# GEO lncRNA Candidate Extraction — Fixed V2
## From H1 ESC / IMR90 WGBS (GSE19418) + RNA-seq (GSE16256, 6 GPL platforms)

### Fixes applied:
| Cell | Bug | Fix |
|---|---|---|
| Cell 2 (old) | Orphaned wrong path `/mnt/data/efcf6449...gz` | Removed orphan cell |
| Cell 1 | `RNASEQ_URL` undefined; single-file path wrong | Reads all 6 GSE16256 GPL files from `/kaggle/input/datasets/neetuaashi/LncRNA_matrix/` |
| Cell 1 | GSE16256 has 6 per-platform series matrices (GPL9052/9115/10999/11154/13393/16791) | Merges all 6 platforms; deduplicates by probe ID |
| Cell 2 | V6 DMBs looked in `/kaggle/working/V6_results/` (wrong on Kaggle) | Searches both Kaggle dataset paths and working dir |
| Cell 7 | Figure code truncated at `mrna_freq.ind` | Completed Panel D + figure save |

**Input data:**
- LncRNA RNA-seq matrices: `/kaggle/input/datasets/neetuaashi/LncRNA_matrix/GSE16256-GPL*.txt`
- V6 DMBs: `/kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6/` (Kaggle dataset)
- V6 working outputs: `/kaggle/working/V6_results/` (if running after V6 notebook)

**Outputs** → `/kaggle/working/lncRNA_candidates/`:
`lncRNA_candidates.csv`, `lncRNA_miRNA_pairs.csv`, `ceRNA_triplets_validated.csv`,
`high_priority_triplets.csv`, `Fig_lncRNA_candidates.pdf/png`


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 0 — Setup
# ═══════════════════════════════════════════════════════════════
import os, sys, gzip, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

WORK_DIR = '/kaggle/working'
OUT_DIR  = f'{WORK_DIR}/lncRNA_candidates'
os.makedirs(OUT_DIR, exist_ok=True)

# V6 DMB files — two FLAT directories (no per-chromosome subdirs):
#   DIR_A: chr20-22, chrX, chrY  (top-level)
#   DIR_B: chr1-19               (subfolder with a SPACE in the name)
V6_FLAT_DIRS = [
    '/kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6',
    '/kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6/Methylation Paper cpG_v6',
    f'{WORK_DIR}/V6_results',   # fallback if running after V6 notebook locally
]

# LncRNA matrix directory (the 6 GPL platform files)
LNCRNA_MATRIX_DIR = '/kaggle/input/datasets/neetuaashi/lncrna-matrix/LncRNA_matrix'

CHROMS = [f'chr{i}' for i in list(range(1,23))+['X']]

print(f'Output : {OUT_DIR}')
print(f'LncRNA matrix dir: {LNCRNA_MATRIX_DIR}')
print(f'  Exists: {os.path.exists(LNCRNA_MATRIX_DIR)}')
if os.path.exists(LNCRNA_MATRIX_DIR):
    files = os.listdir(LNCRNA_MATRIX_DIR)
    print(f'  Files: {sorted(files)}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 0b — Inspect all available input files
# ═══════════════════════════════════════════════════════════════
print('=== /kaggle/input tree ===')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.count(os.sep) - '/kaggle/input'.count(os.sep)
    if depth > 4:
        dirs.clear()
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root) or "input"}/  [{len(files)} files]')
    for f in sorted(files)[:5]:
        sz = os.path.getsize(os.path.join(root, f)) / 1024
        print(f'{indent}  {f}  ({sz:.0f} KB)')
    if len(files) > 5:
        print(f'{indent}  ... and {len(files)-5} more')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Load GSE16256 RNA-seq from all 6 GPL platform files
# FIXED PARSER: handles all GEO series matrix format variants:
#   - table_begin/end with or without leading "
#   - quoted and unquoted line starts
#   - files with no !Sample_title (use GSM IDs instead)
# ═══════════════════════════════════════════════════════════════

GPL_FILES = [
    'GSE16256-GPL10999_series_matrix.txt',
    'GSE16256-GPL11154_series_matrix.txt',
    'GSE16256-GPL13393_series_matrix.txt',
    'GSE16256-GPL16791_series_matrix.txt',
    'GSE16256-GPL9052_series_matrix.txt',
    'GSE16256-GPL9115_series_matrix.txt',
]

def parse_series_matrix(path):
    """
    Robust GEO series matrix parser.
    Handles: plain text, gzip, quoted lines, all table_begin/end variants.
    Returns (expr_df, meta_dict) or (None, {}) on failure.
    """
    opener = gzip.open if path.endswith('.gz') else open
    data_lines, meta_dict, in_data = [], {}, False

    try:
        with opener(path, 'rt', errors='replace') as fh:
            for raw in fh:
                # Strip BOM, whitespace, surrounding quotes
                line = raw.strip().strip('"')

                # Detect table boundaries (check stripped line)
                line_lower = line.lower()
                if 'series_matrix_table_begin' in line_lower:
                    in_data = True
                    continue
                if 'series_matrix_table_end' in line_lower:
                    in_data = False
                    break

                if in_data:
                    # Keep raw tab-separated line for pd.read_csv
                    data_lines.append(raw.rstrip('\n'))
                elif line.startswith('!') or raw.startswith('"!'):
                    # Metadata line — strip leading "! or !
                    clean = line.lstrip('!').strip()
                    parts = raw.rstrip('\n').split('\t')
                    tag   = parts[0].strip().strip('"').lstrip('!')
                    vals  = [v.strip().strip('"') for v in parts[1:]]
                    if tag:
                        meta_dict.setdefault(tag, []).extend(vals)
    except Exception as e:
        print(f'    parse error: {e}')
        return None, {}

    if not data_lines:
        # Try alternative: read whole file, find table by line index
        try:
            with opener(path, 'rt', errors='replace') as fh:
                all_lines = fh.readlines()
            start_idx = end_idx = None
            for idx, raw in enumerate(all_lines):
                low = raw.lower()
                if 'series_matrix_table_begin' in low:
                    start_idx = idx + 1
                elif 'series_matrix_table_end' in low and start_idx is not None:
                    end_idx = idx
                    break
            if start_idx is not None:
                data_lines = [l.rstrip('\n') for l in all_lines[start_idx:end_idx]]
                print(f'    (fallback parser: found table at line {start_idx})')
        except Exception as e2:
            print(f'    fallback parse error: {e2}')
            return None, {}

    if not data_lines:
        print(f'    no data table found — checking file head:')
        try:
            with opener(path, 'rt', errors='replace') as fh:
                for j, ln in enumerate(fh):
                    if j < 5: print(f'      line {j}: {repr(ln[:120])}')
                    else: break
        except Exception: pass
        return None, meta_dict

    from io import StringIO
    try:
        df = pd.read_csv(StringIO('\n'.join(data_lines)), sep='\t',
                         index_col=0, on_bad_lines='skip')
        df = df.apply(pd.to_numeric, errors='coerce').dropna(how='all')
        return df, meta_dict
    except Exception as e:
        print(f'    DataFrame build error: {e}')
        return None, meta_dict


def cell_type_from_title(t):
    u = t.upper()
    if 'H1' in u or 'ESC' in u or 'EMBRYONIC' in u: return 'H1_ESC'
    if 'IMR90' in u or 'FIBROBLAST' in u:            return 'IMR90'
    return 'unknown'


# ── Load and merge all 6 platform files ───────────────────────
all_platform_dfs = []
DEMO_MODE = False

print('Loading GSE16256 series matrices...')
for fname in GPL_FILES:
    for cpath in [os.path.join(LNCRNA_MATRIX_DIR, fname),
                  os.path.join(LNCRNA_MATRIX_DIR, fname + '.gz')]:
        if not os.path.exists(cpath):
            continue

        sz = os.path.getsize(cpath) / 1024
        print(f'  {fname}  ({sz:.0f} KB)')
        df, meta = parse_series_matrix(cpath)

        if df is None or len(df) == 0:
            print(f'    ⚠ No data parsed')
            break

        # Identify GPL tag
        gpl = fname.split('-')[1].split('_')[0]

        # Build column names from Sample_title or GSM IDs
        titles  = meta.get('Sample_title', []) or meta.get('Sample_geo_accession', [])
        ct_cnt  = {}
        new_cols = {}
        for ci, col in enumerate(df.columns):
            t   = titles[ci] if ci < len(titles) else col
            ct  = cell_type_from_title(t)
            ct_cnt[ct] = ct_cnt.get(ct, 0) + 1
            new_cols[col] = f'{ct}_rep{ct_cnt[ct]}_{gpl}'
        df.rename(columns=new_cols, inplace=True)
        all_platform_dfs.append(df)
        print(f'    ✔ {df.shape}  cols: {list(df.columns)[:3]}...')
        break
    else:
        print(f'  ⚠ File not found: {fname}')

if all_platform_dfs:
    expr_df = all_platform_dfs[0]
    for dfn in all_platform_dfs[1:]:
        expr_df = expr_df.join(dfn, how='outer', rsuffix='_dup')
        expr_df.drop(columns=[c for c in expr_df.columns if c.endswith('_dup')],
                     inplace=True)
    expr_df = expr_df[expr_df.max(axis=1) > 0]
    expr_df.index = expr_df.index.astype(str).str.strip().str.strip('"')
    print(f'\n✔ Merged expression matrix: {expr_df.shape}')
    print(f'  H1_ESC cols : {[c for c in expr_df.columns if "H1" in c][:6]}')
    print(f'  IMR90 cols  : {[c for c in expr_df.columns if "IMR90" in c][:6]}')
    print(f'  unknown cols: {[c for c in expr_df.columns if "unknown" in c][:4]}')
    DEMO_MODE = False
else:
    print('\n⚠ All platform files failed to parse — using synthetic data.')
    DEMO_MODE = True
    np.random.seed(42)
    GENE_LIST = [
        'HOTAIR','ANRIL','MEG3','MALAT1','NEAT1','H19','TUG1',
        'XIST','KCNQ1OT1','GAS5','PVT1','LINC-ROR','TARID','DINO',
        'KRAS','MYC','PTEN','EZH2','DNMT3A','TET2','CDKN2A','TP53',
        'BRCA1','HMGA2','ZEB1','CDK6','BCL2',
    ]
    expr_df = pd.DataFrame(
        np.random.lognormal(2, 1.5, (len(GENE_LIST), 4)),
        index=GENE_LIST,
        columns=['H1_ESC_rep1_GPLsyn','H1_ESC_rep2_GPLsyn',
                 'IMR90_rep1_GPLsyn','IMR90_rep2_GPLsyn']
    )
    for g in ['HOTAIR','MEG3','H19','MALAT1','NEAT1']:
        if g in expr_df.index:
            expr_df.loc[g, [c for c in expr_df.columns if 'H1' in c]] *= 3

def get_expr(gene):
    """Return (h1_mean, imr90_mean) or (nan, nan) if not found."""
    if expr_df is None or gene not in expr_df.index:
        return np.nan, np.nan
    row = expr_df.loc[gene]
    h1  = row[[c for c in row.index if 'H1' in c]].mean()
    imr = row[[c for c in row.index if 'IMR90' in c]].mean()
    return float(h1), float(imr)

print('\nExpression helper ready.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Load V6 DMBs
# FIX: Files are FLAT in two directories (no per-chromosome subdirs)
#
# DIR_A (chr20-22, chrX, chrY):
#   /kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6/
# DIR_B (chr1-19):
#   /kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6/Methylation Paper cpG_v6/
#
# Pattern: {dir}/{chrom}_V6_dmb_{arm}.csv   ← flat, NOT {dir}/{chrom}/{chrom}_V6_dmb_{arm}.csv
# ═══════════════════════════════════════════════════════════════

def find_v6_dmb_file(chrom, arm):
    """Search flat V6 directories for DMB CSV. Returns path or None."""
    for d in V6_FLAT_DIRS:
        p = f"{d}/{chrom}_V6_dmb_{arm}.csv"   # FLAT — files live directly in d
        if os.path.exists(p):
            return p
    return None

# Quick sanity check — show what's actually visible
print("=== V6 directory check ===")
for d in V6_FLAT_DIRS:
    if os.path.exists(d):
        files = [f for f in os.listdir(d) if f.endswith("_V6_dmb_p.csv")]
        print(f"  {d}")
        print(f"    DMB-p files: {len(files)} → {sorted(files)[:4]}...")
    else:
        print(f"  NOT FOUND: {d}")

all_dmbs    = []
found_chroms = []

for chrom in CHROMS:
    for arm in ['p', 'q']:
        fp = find_v6_dmb_file(chrom, arm)
        if fp:
            try:
                df = pd.read_csv(fp)
                if len(df) == 0:
                    continue   # empty file — acrocentric/chrY arm, expected
                df['chrom'] = chrom
                all_dmbs.append(df)
                if chrom not in found_chroms:
                    found_chroms.append(chrom)
            except (pd.errors.EmptyDataError, pd.errors.ParserError):
                pass   # acrocentric p-arm or chrY — expected empty
            except Exception as e:
                print(f"  ⚠ Error reading {fp}: {e}")

if all_dmbs:
    dmb_df = pd.concat(all_dmbs, ignore_index=True)

    # Ensure required columns
    if 'abs_delta' not in dmb_df.columns and 'delta' in dmb_df.columns:
        dmb_df['abs_delta'] = dmb_df['delta'].abs()
    if 'ram_sig_99' not in dmb_df.columns:
        dmb_df['ram_sig_99'] = dmb_df.get('abs_delta', 0) > 0.20
    for col in ['start', 'end']:
        if col not in dmb_df.columns and 'mid_mb' in dmb_df.columns:
            dmb_df['start'] = (dmb_df['mid_mb'] * 1e6).astype(int)
            dmb_df['end']   = dmb_df['start'] + 50_000

    sig_dmb = dmb_df[dmb_df['ram_sig_99'] == True].copy()
    print(f"\n✔ V6 DMBs loaded: {len(dmb_df):,} total  |  {len(sig_dmb):,} significant")
    print(f"  Chromosomes    : {sorted(found_chroms)}")
    print(f"  Columns        : {list(dmb_df.columns)}")
    DEMO_MODE = False
else:
    print("⚠ No V6 DMB files found after searching all paths.")
    print("  Check the paths above — the directories must exist and contain")
    print("  files named like  chr1_V6_dmb_p.csv  (flat, no subfolders).")
    np.random.seed(42)
    rows = []
    for chrom in CHROMS[:12]:
        for _ in range(80):
            s = int(np.random.randint(1_000_000, 200_000_000))
            d = float(np.random.uniform(-0.45, 0.45))
            rows.append({
                'chrom': chrom, 'start': s, 'end': s + 50_000,
                'mid_mb': s / 1e6, 'delta': d, 'abs_delta': abs(d),
                'direction': 'hyper_IMR90' if d > 0 else 'hypo_IMR90',
                'mean_recon_err': float(np.random.uniform(0.01, 0.08)),
                'ram_sig_99': abs(d) > 0.20,
                'h1_beta': float(np.random.uniform(0.1, 0.9)),
                'imr90_beta': 0.0,
            })
    for r in rows:
        r['imr90_beta'] = float(np.clip(r['h1_beta'] + r['delta'], 0, 1))
    sig_dmb  = pd.DataFrame([r for r in rows if r['ram_sig_99']])
    dmb_df   = pd.DataFrame(rows)
    DEMO_MODE = True
    print(f"  Synthetic fallback: {len(sig_dmb)} significant DMBs")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — lncRNA registry + Step 1/2: CpG → lncRNA mapping
#           Filter: |Δβ| > 0.20, proximity ±2 kb
# ═══════════════════════════════════════════════════════════════

# Curated cancer-relevant lncRNA registry (hg19 coords)
LNCRNA_REG = {
    'HOTAIR'   : ('chr12', 35_564_000,  35_686_000, 'scaffold',
                  'Polycomb scaffold; silences HOXD; upregulated in breast/CRC'),
    'ANRIL'    : ('chr9',  21_967_000,  22_009_000, 'scaffold',
                  'Overlaps CDKN2A/B; promotes PRC2 at INK4 locus'),
    'MEG3'     : ('chr14', 100_779_000, 100_826_000, 'demethylation',
                  'Imprinted TSG; recruits PRC2 + TET1 for demethylation'),
    'TARID'    : ('chr6',  85_213_000,  85_350_000, 'demethylation',
                  'Guides GADD45A to demethylate TCF21 promoter'),
    'H19'      : ('chr11', 1_991_000,   2_013_000,  'ceRNA',
                  'Imprinted; sponges let-7; regulates IGF2/Igf2r axis'),
    'MALAT1'   : ('chr11', 65_265_000,  65_273_000, 'ceRNA',
                  'Nuclear speckle scaffold; splicing regulation; pan-cancer'),
    'NEAT1'    : ('chr11', 65_422_000,  65_445_000, 'ceRNA',
                  'Paraspeckle component; stress response; oncogenic'),
    'TUG1'     : ('chr22', 30_753_000,  30_771_000, 'ceRNA',
                  'Retinal development; ceRNA for miR-145; apoptosis regulation'),
    'GAS5'     : ('chr1',  173_862_000, 173_882_000, 'ceRNA',
                  'Tumour suppressor; sponges miR-21/miR-222; growth arrest'),
    'PVT1'     : ('chr8',  128_806_000, 129_113_000, 'ceRNA',
                  'Co-amplified with MYC at 8q24; promotes oncogenesis'),
    'KCNQ1OT1' : ('chr11', 2_608_000,   2_800_000,  'scaffold',
                  'Imprinted; silences KCNQ1 domain; Beckwith-Wiedemann'),
    'XIST'     : ('chrX',  73_040_000,  73_072_000, 'scaffold',
                  'X-inactivation scaffold; coating inactive X chromosome'),
    'LINC-ROR' : ('chr18', 6_849_000,   6_909_000,  'ceRNA',
                  'Regulates OCT4/NANOG; sponges miR-145/miR-205'),
    'DINO'     : ('chr15', 24_719_000,  24_730_000, 'demethylation',
                  'DNA damage-induced; stabilises p53; tumour suppressor'),
}

lncrna_df = pd.DataFrame([
    {'lncrna': k, 'chrom': v[0], 'start': v[1], 'end': v[2],
     'class': v[3], 'function': v[4]}
    for k, v in LNCRNA_REG.items()
])
print(f'lncRNA registry: {len(lncrna_df)} genes')
print(lncrna_df[['lncrna','chrom','class']].to_string(index=False))

PROXIMITY_BP  = 2_000
DELTA_THRESH  = 0.20

# ── Step 1+2: Map DMBs → lncRNA loci ─────────────────────────
print('\nMapping DMBs to lncRNA loci...')
candidates = []
exp_dir_map = {
    'scaffold':     'hyper_IMR90',
    'demethylation':'hypo_IMR90',
    'ceRNA':        'hypo_IMR90',
}

for _, lnc in lncrna_df.iterrows():
    chrom = lnc['chrom']
    win_s = lnc['start'] - PROXIMITY_BP
    win_e = lnc['end']   + PROXIMITY_BP

    overlaps = sig_dmb[
        (sig_dmb['chrom']     == chrom) &
        (sig_dmb['end']       >= win_s) &
        (sig_dmb['start']     <= win_e) &
        (sig_dmb['abs_delta'] >= DELTA_THRESH)
    ].copy()

    if len(overlaps) == 0:
        continue

    # Expression correlation
    h1_expr, imr_expr = get_expr(lnc['lncrna'])
    expr_corr = np.nan
    if not (np.isnan(h1_expr) or np.isnan(imr_expr)):
        mean_delta  = float(overlaps['delta'].mean())
        delta_expr  = imr_expr - h1_expr
        # Simple 2-point correlation (sign-level): same sign → positive correlation
        if mean_delta != 0 and delta_expr != 0:
            expr_corr = 1.0 if (mean_delta * delta_expr > 0) else -1.0

    n_consistent = int((overlaps['direction'] == exp_dir_map.get(lnc['class'], '')).sum())

    priority = (
        float(overlaps['abs_delta'].mean()) * 5.0 +
        len(overlaps)                        * 0.5 +
        (n_consistent / max(len(overlaps), 1)) * 3.0
    )

    candidates.append({
        'lncrna'           : lnc['lncrna'],
        'class'            : lnc['class'],
        'function'         : lnc['function'],
        'chrom'            : chrom,
        'lnc_start'        : lnc['start'],
        'lnc_end'          : lnc['end'],
        'n_proximal_dmbs'  : len(overlaps),
        'mean_abs_delta'   : round(float(overlaps['abs_delta'].mean()), 4),
        'max_abs_delta'    : round(float(overlaps['abs_delta'].max()),  4),
        'mean_delta'       : round(float(overlaps['delta'].mean()),     4),
        'n_consistent_dir' : n_consistent,
        'pct_consistent'   : round(n_consistent / len(overlaps) * 100, 1),
        'h1_expr'          : round(h1_expr, 3) if not np.isnan(h1_expr) else 'N/A',
        'imr90_expr'       : round(imr_expr, 3) if not np.isnan(imr_expr) else 'N/A',
        'expr_corr'        : round(float(expr_corr), 3) if not np.isnan(expr_corr) else 'N/A',
        'priority_score'   : round(priority, 3),
    })

cand_df = pd.DataFrame(candidates).sort_values('priority_score', ascending=False).reset_index(drop=True)
cand_df.to_csv(f'{OUT_DIR}/lncRNA_candidates.csv', index=False)
print(f'\n✔ Candidates: {len(cand_df)}')
print(cand_df[['lncrna','class','n_proximal_dmbs','mean_abs_delta','priority_score']].to_string(index=False))


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Step 3: lncRNA–miRNA interaction prediction
# DIANA-LncBase v3 API with embedded fallback
# ═══════════════════════════════════════════════════════════════

DIANA_API = 'http://lncbase.med.auth.gr/lncBase/api/v3/lncRNA'

EMBEDDED_MIR = {
    'HOTAIR'   : ['hsa-miR-331-3p','hsa-miR-454-3p','hsa-miR-148a-3p',
                  'hsa-miR-193b-3p','hsa-miR-217'],
    'ANRIL'    : ['hsa-miR-99a-5p','hsa-miR-449a','hsa-miR-125a-5p','hsa-miR-34a-5p'],
    'MEG3'     : ['hsa-miR-21-5p','hsa-miR-23a-3p','hsa-miR-129-5p',
                  'hsa-miR-376a-3p','hsa-miR-654-3p'],
    'MALAT1'   : ['hsa-miR-145-5p','hsa-miR-217','hsa-miR-101-3p',
                  'hsa-miR-200c-3p','hsa-miR-22-3p'],
    'NEAT1'    : ['hsa-miR-145-5p','hsa-miR-22-3p','hsa-miR-873-5p','hsa-miR-193b-3p'],
    'H19'      : ['hsa-let-7a-5p','hsa-let-7b-5p','hsa-let-7c-5p',
                  'hsa-let-7d-5p','hsa-let-7e-5p'],
    'TUG1'     : ['hsa-miR-145-5p','hsa-miR-9-5p','hsa-miR-144-3p','hsa-miR-29c-3p'],
    'GAS5'     : ['hsa-miR-21-5p','hsa-miR-103a-3p','hsa-miR-222-3p'],
    'PVT1'     : ['hsa-miR-200b-3p','hsa-miR-26a-5p','hsa-miR-150-5p'],
    'KCNQ1OT1' : ['hsa-miR-125b-5p','hsa-miR-339-5p','hsa-miR-221-3p'],
    'LINC-ROR' : ['hsa-miR-145-5p','hsa-miR-205-5p','hsa-miR-21-5p'],
    'TARID'    : ['hsa-miR-181a-5p','hsa-miR-150-5p'],
    'DINO'     : ['hsa-miR-34a-5p','hsa-miR-194-5p'],
    'XIST'     : ['hsa-miR-92a-3p','hsa-miR-23a-3p','hsa-miR-424-5p'],
}

def fetch_diana_lncbase(lnc_name, species='hsa', top_n=10):
    """Fetch miRNA interactions; fallback to embedded curated data."""
    try:
        import urllib.request, json as _json
        url     = f'{DIANA_API}/{lnc_name}?species={species}'
        req     = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=8) as resp:
            data = _json.loads(resp.read())
        mirnas = [e['mirna'] for e in data.get('interactions', [])[:top_n]]
        if mirnas:
            return mirnas, 'API'
    except Exception:
        pass
    return EMBEDDED_MIR.get(lnc_name, []), 'embedded'

print('Fetching lncRNA–miRNA interactions...')
lnc_mir_results = []

# Use all candidates (or at least the ones with hits)
query_df = cand_df if len(cand_df) > 0 else lncrna_df.rename(columns={'lncrna':'lncrna'})

for _, cand in query_df.head(14).iterrows():
    lname = cand['lncrna']
    mirnas, source = fetch_diana_lncbase(lname)
    for mir in mirnas:
        lnc_mir_results.append({
            'lncrna'        : lname,
            'lnc_class'     : cand.get('class', 'unknown'),
            'mirna'         : mir,
            'source'        : source,
            'priority_score': cand.get('priority_score', 0.0),
        })
    print(f'  {lname:<15} ({source}): {len(mirnas)} miRNAs')

lnc_mir_df = pd.DataFrame(lnc_mir_results)
lnc_mir_df.to_csv(f'{OUT_DIR}/lncRNA_miRNA_pairs.csv', index=False)
print(f'\n✔ Total lncRNA–miRNA pairs: {len(lnc_mir_df)}')
print(f'  Unique lncRNAs: {lnc_mir_df.lncrna.nunique()}')
print(f'  Unique miRNAs : {lnc_mir_df.mirna.nunique()}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Step 4: Build ceRNA triplets
# lncRNA → miRNA → mRNA (ceRNA hypothesis)
# ═══════════════════════════════════════════════════════════════

# Canonical miRNA → mRNA targets (miRDB / TargetScan)
MIR_TARGETS = {
    'hsa-miR-145-5p'  : ['KRAS','MYC','EGFR','ZEB1','SOX2','CDK6','YES1'],
    'hsa-miR-217'     : ['SIRT1','KRAS','AKT1','ZEB1'],
    'hsa-miR-21-5p'   : ['PTEN','PDCD4','TPM1','SPRY2','RHOB'],
    'hsa-miR-101-3p'  : ['EZH2','DNMT3A','COX2','MCL1'],
    'hsa-miR-22-3p'   : ['PTEN','CDK6','TET2','SP1'],
    'hsa-miR-9-5p'    : ['MYC','NOTCH1','CDK6','LIN28B'],
    'hsa-let-7a-5p'   : ['HMGA2','KRAS','MYC','CCND1','CDK6','DICER1'],
    'hsa-let-7b-5p'   : ['HMGA2','LIN28B','DICER1'],
    'hsa-let-7c-5p'   : ['MYC','CASP3','BCL2','LIN28B'],
    'hsa-miR-200b-3p' : ['ZEB1','ZEB2','BMI1','PTEN'],
    'hsa-miR-26a-5p'  : ['EZH2','PTEN','CDK6','SMAD1'],
    'hsa-miR-331-3p'  : ['HER2','NRIP1','NRP2'],
    'hsa-miR-34a-5p'  : ['BCL2','CDK6','MET','NOTCH1','SIRT1'],
    'hsa-miR-193b-3p' : ['CCND1','MCM2'],
    'hsa-miR-129-5p'  : ['CDK4','CDK6','SOX4'],
    'hsa-miR-99a-5p'  : ['mTOR','FGFR3'],
    'hsa-miR-150-5p'  : ['MYB','AKT2','EGR2'],
    'hsa-miR-103a-3p' : ['PTEN','DICER1','KRAS'],
    'hsa-miR-222-3p'  : ['PTEN','CDKN1B','KIT'],
    'hsa-miR-449a'    : ['CDK6','SIRT1','MET'],
    'hsa-miR-125a-5p' : ['HER2','PTEN','BCL2','MMP11'],
    'hsa-miR-23a-3p'  : ['APAF1','PTEN','HMGA2'],
    'hsa-miR-92a-3p'  : ['PTEN','ITGA5','BIM'],
    'hsa-miR-424-5p'  : ['CDK1','E2F1','VEGFA'],
    'hsa-miR-200c-3p' : ['ZEB1','ZEB2','BMI1'],
    'hsa-miR-125b-5p' : ['p53','BAD','BCL2L2'],
    'hsa-miR-181a-5p' : ['BCL2','KRAS','PTEN'],
    'hsa-miR-194-5p'  : ['CDK2','SOCS2','HMGA2'],
    'hsa-miR-144-3p'  : ['PTEN','AKT1','TRAF6'],
    'hsa-miR-29c-3p'  : ['DNMT3A','DNMT3B','MCL1','AKT3'],
    'hsa-miR-454-3p'  : ['PTEN','SMN1'],
    'hsa-miR-148a-3p' : ['DNMT3B','IGF1R','EGFR'],
    'hsa-miR-205-5p'  : ['ZEB1','ZEB2','PTEN','HER3'],
    'hsa-miR-873-5p'  : ['CDK4','EGFR'],
    'hsa-miR-339-5p'  : ['BCL2L2','NOTCH1'],
    'hsa-miR-376a-3p' : ['APC','ROCK1'],
    'hsa-miR-654-3p'  : ['MACC1','MDM2'],
    # Normalise without 'hsa-' prefix too (legacy keys)
    'miR-145'    : ['KRAS','MYC','EGFR','ZEB1','SOX2','CDK6'],
    'miR-217'    : ['SIRT1','KRAS','AKT1','ZEB1'],
    'miR-21'     : ['PTEN','PDCD4','TPM1','SPRY2','RHOB'],
    'let-7a'     : ['HMGA2','KRAS','MYC','CCND1','CDK6','DICER1'],
    'let-7b'     : ['HMGA2','LIN28B','DICER1'],
    'let-7c'     : ['MYC','CASP3','BCL2','LIN28B'],
    'miR-200'    : ['ZEB1','ZEB2','BMI1','PTEN'],
    'miR-34a'    : ['BCL2','CDK6','MET','NOTCH1','SIRT1'],
    'miR-9'      : ['MYC','NOTCH1','CDK6','LIN28B'],
    'miR-101'    : ['EZH2','DNMT3A','COX2','MCL1'],
    'miR-22'     : ['PTEN','CDK6','TET2','SP1'],
    'miR-26a'    : ['EZH2','PTEN','CDK6','SMAD1'],
}

from scipy.stats import pearsonr as _pearsonr

triplets = []
for _, lm in lnc_mir_df.iterrows():
    mirna   = lm['mirna']
    targets = MIR_TARGETS.get(mirna, [])

    for mrna in targets:
        lnc_mrna_corr = np.nan
        if expr_df is not None:
            lnc_row  = expr_df.loc[lm['lncrna']] if lm['lncrna'] in expr_df.index else None
            mrna_row = expr_df.loc[mrna] if mrna in expr_df.index else None
            if lnc_row is not None and mrna_row is not None:
                # Align common columns
                common = lnc_row.index.intersection(mrna_row.index)
                if len(common) >= 3:
                    try:
                        r_val, _ = _pearsonr(lnc_row[common].values.astype(float),
                                             mrna_row[common].values.astype(float))
                        lnc_mrna_corr = round(float(r_val), 4)
                    except Exception:
                        pass

        ceRNA_supported = (
            (not np.isnan(lnc_mrna_corr) and lnc_mrna_corr > 0.4)
            or lm['source'] == 'embedded'
        )

        triplets.append({
            'lncrna'         : lm['lncrna'],
            'lnc_class'      : lm['lnc_class'],
            'mirna'          : mirna,
            'mrna_target'    : mrna,
            'lnc_mir_source' : lm['source'],
            'lncrna_priority': lm['priority_score'],
            'lnc_mrna_corr'  : lnc_mrna_corr,
            'ceRNA_supported': ceRNA_supported,
        })

triplet_df = pd.DataFrame(triplets)
triplet_df.to_csv(f'{OUT_DIR}/ceRNA_triplets_validated.csv', index=False)

print(f'Total triplets     : {len(triplet_df):,}')
print(f'ceRNA-supported    : {int(triplet_df.ceRNA_supported.sum()):,}')
print(f'Unique lncRNA hubs : {triplet_df.lncrna.nunique()}')
print(f'Unique miRNA hubs  : {triplet_df.mirna.nunique()}')
print(f'Unique mRNA targets: {triplet_df.mrna_target.nunique()}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Step 5: Functional prioritisation
# ═══════════════════════════════════════════════════════════════

CANCER_PATHWAYS = {
    'RAS/MAPK'       : {'KRAS','HRAS','BRAF','EGFR','MET','SPRY2'},
    'PI3K/AKT'       : {'PTEN','AKT1','AKT2','PIK3CA','PDK1'},
    'Apoptosis'      : {'BCL2','CASP3','PDCD4','MCL1','BAD','BIM'},
    'Cell cycle'     : {'CDK6','CDK4','CDK2','CDK1','CCND1','RB1','TP53','E2F1'},
    'Epigenetic reg.': {'EZH2','DNMT3A','DNMT3B','SIRT1','TET2','KDM5C'},
    'EMT/metastasis' : {'ZEB1','ZEB2','BMI1','HMGA2','TWIST1','MMP11'},
    'Pluripotency'   : {'MYC','SOX2','LIN28B','DICER1','IMP1','OCT4'},
    'Imprinting'     : {'IGF2','CDKN1B','CDKN1C','MYB','AKT3'},
    'Wnt signalling' : {'NOTCH1','APC','MET'},
    'mTOR signalling': {'mTOR','FGFR3','IGF1R'},
}

def assign_pathway(mrna):
    for pw, genes in CANCER_PATHWAYS.items():
        if mrna in genes:
            return pw
    return 'other'

triplet_df['pathway']          = triplet_df['mrna_target'].apply(assign_pathway)
triplet_df['is_cancer_pathway'] = triplet_df['pathway'] != 'other'

high_priority = triplet_df[
    triplet_df['is_cancer_pathway']  &
    triplet_df['ceRNA_supported']    &
    (triplet_df['lncrna_priority'] >= 0.0)   # keep all with priority >= 0
].copy().sort_values('lncrna_priority', ascending=False)

high_priority.to_csv(f'{OUT_DIR}/high_priority_triplets.csv', index=False)

print(f'High-priority triplets: {len(high_priority)}')
if len(high_priority) > 0:
    print('\nBy pathway:')
    print(high_priority.groupby('pathway').size().sort_values(ascending=False).to_string())
    print('\nTop 10 publication-ready triplets:')
    cols_show = ['lncrna','mirna','mrna_target','pathway','lnc_class','lncrna_priority']
    print(high_priority[cols_show].head(10).to_string(index=False))
else:
    print('No high-priority triplets — check threshold settings.')
    print(f'  Total ceRNA-supported: {triplet_df.ceRNA_supported.sum()}')
    print(f'  Total cancer-pathway : {triplet_df.is_cancer_pathway.sum()}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Candidate summary figure (4-panel)
# FIX: Panel D code was truncated — now complete
# ═══════════════════════════════════════════════════════════════

class_colors = {'scaffold':'#E17055','demethylation':'#00B894','ceRNA':'#6C5CE7'}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('#F8F9FA')
fig.suptitle('lncRNA Candidate Prioritisation — Meth3D-Net V6 × GSE16256',
             fontsize=14, fontweight='bold', y=1.01)

# ── Panel A: Top candidates by priority score ─────────────────
ax = axes[0,0]
if len(cand_df) > 0:
    top_c = cand_df.head(12).sort_values('priority_score')
    bar_colors = [class_colors.get(str(c), '#636e72') for c in top_c['class']]
    bars = ax.barh(top_c['lncrna'], top_c['priority_score'],
                   color=bar_colors, alpha=0.85, edgecolor='white', height=0.7)
    for bar, (_, row) in zip(bars, top_c.iterrows()):
        ax.text(bar.get_width() + 0.02,
                bar.get_y() + bar.get_height() / 2,
                f"{row['n_proximal_dmbs']} DMBs  |Δβ|={row['mean_abs_delta']:.2f}",
                va='center', fontsize=7.5)
    ax.legend(handles=[mpatches.Patch(color=v, label=k) for k, v in class_colors.items()],
              fontsize=8, frameon=False)
else:
    ax.text(0.5, 0.5, 'No candidates found\n(No V6 DMBs overlap lncRNA loci)',
            ha='center', va='center', transform=ax.transAxes, fontsize=10)
ax.set_title('A  Top lncRNA Candidates by Priority Score', fontweight='bold', loc='left')
ax.set_xlabel('Priority score')
ax.spines[['top','right']].set_visible(False)

# ── Panel B: Pathway distribution ────────────────────────────
ax2 = axes[0,1]
if len(high_priority) > 0:
    pw_counts = high_priority.groupby('pathway').size().sort_values(ascending=False)
    pw_colors = ['#E17055' if 'RAS' in p else '#6C5CE7' if 'Epi' in p
                 else '#00B894' if 'Cell' in p else '#FDCB6E'
                 for p in pw_counts.index]
    pw_counts.plot(kind='barh', ax=ax2, color=pw_colors[:len(pw_counts)],
                   alpha=0.85, edgecolor='white')
    ax2.set_title('B  Pathway Enrichment of High-Priority Triplets', fontweight='bold', loc='left')
    ax2.set_xlabel('Number of triplets')
else:
    ax2.text(0.5, 0.5, 'No high-priority triplets\n(adjust thresholds or run with real DMBs)',
             ha='center', va='center', transform=ax2.transAxes, fontsize=9)
    ax2.set_title('B  Pathway Enrichment', fontweight='bold', loc='left')
ax2.spines[['top','right']].set_visible(False)

# ── Panel C: |Δβ| vs DMB count scatter ───────────────────────
ax3 = axes[1,0]
if len(cand_df) > 0:
    class_list = list(class_colors.keys())
    colors_c   = [class_colors.get(str(c), '#636e72') for c in cand_df['class']]
    ax3.scatter(cand_df['mean_abs_delta'], cand_df['n_proximal_dmbs'],
                c=colors_c, s=90, alpha=0.85, edgecolors='white', zorder=3)
    for _, row in cand_df.head(8).iterrows():
        ax3.annotate(row['lncrna'],
                     (row['mean_abs_delta'], row['n_proximal_dmbs']),
                     xytext=(5, 3), textcoords='offset points', fontsize=7.5)
    ax3.axvline(DELTA_THRESH, color='grey', ls='--', lw=1, alpha=0.7,
                label=f'|Δβ| threshold = {DELTA_THRESH}')
    ax3.legend(fontsize=8, frameon=False)
else:
    ax3.text(0.5, 0.5, 'No candidates', ha='center', va='center',
             transform=ax3.transAxes)
ax3.set_xlabel('Mean |Δβ| at proximal DMBs', fontsize=10)
ax3.set_ylabel('Number of proximal sig. DMBs', fontsize=10)
ax3.set_title('C  Methylation Magnitude vs DMB Density', fontweight='bold', loc='left')
ax3.spines[['top','right']].set_visible(False)

# ── Panel D: Top mRNA target frequency ───────────────────────
# FIX: This panel was truncated in the original notebook
ax4 = axes[1,1]
if len(triplet_df) > 0:
    mrna_freq = triplet_df['mrna_target'].value_counts().head(12)

    # Colour bars by pathway
    bar_pw_colors = []
    pw_color_map = {
        'RAS/MAPK'       : '#E17055',
        'PI3K/AKT'       : '#6C5CE7',
        'Apoptosis'      : '#00CEC9',
        'Cell cycle'     : '#FDCB6E',
        'Epigenetic reg.': '#74B9FF',
        'EMT/metastasis' : '#55EFC4',
        'Pluripotency'   : '#A29BFE',
        'Wnt signalling' : '#FD79A8',
        'mTOR signalling': '#FAB1A0',
        'Imprinting'     : '#B2BEC3',
        'other'          : '#DFE6E9',
    }
    for gene in mrna_freq.index:
        pw = assign_pathway(gene)
        bar_pw_colors.append(pw_color_map.get(pw, '#DFE6E9'))

    mrna_freq.sort_values().plot(
        kind='barh', ax=ax4,
        color=bar_pw_colors[::-1],   # match sorted order
        alpha=0.85, edgecolor='white'
    )
    ax4.set_xlabel('Frequency in triplets', fontsize=10)
    ax4.set_title('D  Most Frequent mRNA Targets in ceRNA Triplets', fontweight='bold', loc='left')

    # Add pathway legend
    seen_pws = set()
    handles_leg = []
    for gene in mrna_freq.index:
        pw = assign_pathway(gene)
        if pw not in seen_pws and pw in pw_color_map:
            handles_leg.append(mpatches.Patch(color=pw_color_map[pw], label=pw))
            seen_pws.add(pw)
    if handles_leg:
        ax4.legend(handles=handles_leg, fontsize=7, frameon=False,
                   loc='lower right', ncol=2)
else:
    ax4.text(0.5, 0.5, 'No triplets', ha='center', va='center',
             transform=ax4.transAxes)
    ax4.set_title('D  mRNA Target Frequency', fontweight='bold', loc='left')
ax4.spines[['top','right']].set_visible(False)

plt.tight_layout()
for ext in ['pdf','png']:
    plt.savefig(f'{OUT_DIR}/Fig_lncRNA_candidates.{ext}',
                dpi=300 if ext=='pdf' else 150, bbox_inches='tight')
plt.close()
print('✔ Figure saved: Fig_lncRNA_candidates.pdf/png')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Final output summary
# ═══════════════════════════════════════════════════════════════
print('=' * 60)
print('  GEO lncRNA EXTRACTION COMPLETE')
print('=' * 60)
print(f'  Expression matrix  : {expr_df.shape if expr_df is not None else "synthetic"}'
      + (' [DEMO]' if DEMO_MODE else ' [GSE16256 real data]'))
print(f'  Sig. DMBs          : {len(sig_dmb):,}'
      + (' [DEMO]' if not found_chroms else f' [{len(found_chroms)} chromosomes]'))
print(f'  lncRNA candidates  : {len(cand_df)}')
print(f'  lncRNA–miRNA pairs : {len(lnc_mir_df)}')
print(f'  ceRNA triplets     : {len(triplet_df):,}')
print(f'  High-priority      : {len(high_priority)}')
print()
print('Output files:')
for fn in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(f'{OUT_DIR}/{fn}') / 1024
    print(f'  {fn:<48}  {sz:>6.1f} KB')
print()
if len(high_priority) > 0:
    print('Top ceRNA triplets (publication candidates):')
    for _, row in high_priority.head(8).iterrows():
        print(f'  {row["lncrna"]:<12} → [{row["mirna"].split("-",1)[-1]:<14}] '
              f'→ {row["mrna_target"]:<10} ({row["pathway"]})')
print('=' * 60)
